# **1.02 - Visual Feature Extraction**

## **Visual Feature Extraction**

Used keywords specified in **visual_keywords.txt**. 

Keywords generated with 2 queries to chatgpt:
 - "could you generate a long list of verbs that are similar to "saw", "viewed", and "spot" in all tenses."

 - "could you generate a long list of nouns similar to "picture" "images" "drawings" etc"
 
 - Keywords are stored in *keywords.json* under paths:
    - keywords.Verbs.Visual
    - keywords.Nouns.Visual
 
**"Visual Evidence" [bool]**

- True - visual evidence in description
    - example True flag {index: 53}: *'Many people claim to see a light come out of river and chase their vehicle to the end of the road'*

- False - otherwise
    - example False flag {index: 4}: *'Kappa Delta Sorority - The Kappa Delta Sorority is haunted by an entity simply known as \'P\'. It is said she was a sister there who died in a car accident. Current sisters there have reported hearing giggling and running around coming from the upstairs floor while they are in the basement. At one time a sister called out to "P" and received a "hello" in reply.'*

### Feature Extraction

In [ ]:
import pandas as pd
import re 
import time
import json

# Outfile CSV
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading CSV
df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

# Feature Names
feature_names = ["Visual_Evidence"]


# Define Audio Keyword List
keywords = json.load(open("../data/keywords/keywords.json"))
visual_keywords = keywords["Verbs"]["Visual"] + keywords["Nouns"]["Visual"] 


def contains_visual_keywords(text):
    if isinstance(text, str):
        for keyword in visual_keywords:
            if re.search(keyword, text, re.IGNORECASE):
                return True  
    return False

start = time.time()
df["Visual_Evidence"] = df["description"].apply(contains_visual_keywords)
end = time.time()


### Post Processing

In [6]:
## Get total counts and format for printout ##
counts = df["Visual_Evidence"].value_counts()
true_counts = counts.get(True, 0)
false_counts = counts.get(False, 0)

extract_printout = [("True Counts", true_counts), 
                    ("False Counts", false_counts)]

max_len = max(len(category) for category, _ in extract_printout)  

### Report and Save

In [7]:
## Printout Report ##
print("-" * 150, "Extraction Completed", "-" * 150)
print(f"Extraction Took: {end - start:.6f} seconds", end = "\n\n")
print("\n".join([f"{category.ljust(max_len)}| {count}" for category, count in extract_printout]))
print("-" * 150)

print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep = "\t")

# Check if feature exists
for feature in feature_names:
    
    # If it exists, update values
    if feature in out_df.columns:
        out_df[feature].update(df[feature].values)
        out_df[feature] = df[feature].values

    # If not add entire column
    else:
        out_df[feature] = df[feature].values

out_df.to_csv(f"{outfile}", sep = "\t", index = False)

print(f"CSV Saved to {outfile}")

------------------------------------------------------------------------------------------------------------------------------------------------------ Extraction Completed ------------------------------------------------------------------------------------------------------------------------------------------------------
Extraction Took: 1.209420 seconds

True Counts | 7208
False Counts| 3783
------------------------------------------------------------------------------------------------------------------------------------------------------
Saving to CSV...


/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_92093/512019962.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  out_df[feature].update(df[feature].values)


CSV Saved to ../data/processed/haunted_places_features_added.tab
